In [ ]:
import os
import smtplib
import pandas as pd
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
import locale

# Configurar idioma español (usa 'es_ES' o 'es_CO' según tu sistema)
try:
    locale.setlocale(locale.LC_TIME, 'es_CO.utf8')  # Español Colombia
except:
    locale.setlocale(locale.LC_TIME, 'es_ES.utf8')  # Español España

# === CONFIGURACIÓN DE CORREO ===
SENDER_EMAIL = "wcastro@lapocion.com"
SENDER_PASSWORD =  # Contraseña de aplicación de Gmail
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465

# === CONFIGURACIÓN DE ARCHIVOS ===
ruta_excel = r"C:\Users\Dataa\Desktop\empleados.xlsx"
columna_estado = "estado"

# === FUNCIÓN PARA ENVIAR CORREO CON ADJUNTO ===
def enviar_correo_con_pdf(destinatario_email, nombre_empleado, ruta_pdf, fecha):
    """
    Envía un correo con el desprendible de pago adjunto.
    
    Args:
        destinatario_email: Email del empleado
        nombre_empleado: Nombre del empleado
        ruta_pdf: Ruta completa al archivo PDF
    
    Returns:
        tuple: (éxito: bool, mensaje_error: str)
    """
    try:
        # Crear el mensaje
        message = MIMEMultipart()
        message["From"] = SENDER_EMAIL
        message["To"] = destinatario_email
        message["Subject"] = f"Desprendible de Pago {fecha} - {nombre_empleado}"
        
        # # Cuerpo del correo en texto plano
        # texto_plano = f"""Hola {nombre_empleado},


        
        # Cuerpo del correo en HTML (más profesional)
        texto_html = f"""
        <html>
        <body style="font-family: 'Poppins', Arial, sans-serif; background-color: #f7f5ff; color: #333; margin: 0; padding: 0;">
            <div style="max-width: 600px; margin: 40px auto; background: #ffffff; border-radius: 12px; padding: 25px; box-shadow: 0 4px 15px rgba(0,0,0,0.08); text-align: center;">
            
            <h1 style="color: #6c2bd9; font-size: 26px; margin-bottom: 10px;">
                💜 ¡Sííí! ¡Pagaron en <strong>La Poción</strong>! 💰✨
            </h1>

            <p style="font-size: 16px; color: #444;">
                Hola <strong>{nombre_empleado}</strong> 🧙‍♀️,
            </p>

            <p style="font-size: 16px; margin: 15px 0;">
                ¡El momento mágico del mes ha llegado! 💫  
                En el adjunto encontrarás tu <strong>desprendible de pago</strong> para que lo revises tranquilo.
            </p>

            <p style="font-size: 16px; margin: 15px 0;">
                Gracias por seguir poniendo tu energía y buena vibra cada día en <strong>La Poción</strong>.  
                ¡Juntos seguimos creando magia! 🪄✨
            </p>

            <div style="margin: 25px 0;">
                <img src="https://cdn-icons-png.flaticon.com/512/2729/2729007.png" alt="Icono de magia" width="100" style="opacity:0.9;">
            </div>

            <hr style="border: 1px solid #eee; margin: 30px 0;">

            <p style="color: #9b59b6; font-size: 14px; font-style: italic; margin: 20px 0;">
                ✨ <strong>Porque ser tú te hace increíble 💜</strong> ✨
            </p>

            <p style="color: #9b59b6; font-size: 13px;">
                <em>Este mensaje fue enviado automáticamente por el caldero encantado de Recursos Humanos 🔮<br>
                No respondas directamente, o podrías despertar a los duendes contables 😅</em>
            </p>

            <p style="color: #555; font-size: 13px; margin-top: 10px;">
                Con cariño,<br>
                <strong>💼 Equipo de Recursos Humanos - La Pocion</strong>
            </p>
            </div>
        </body>
        </html>
        """



        
        # Adjuntar ambas versiones del mensaje
        # message.attach(MIMEText(texto_plano, "plain", "utf-8"))
        message.attach(MIMEText(texto_html, "html", "utf-8"))
        
        # Adjuntar el archivo PDF
        nombre_archivo = os.path.basename(ruta_pdf)
        
        with open(ruta_pdf, "rb") as archivo:
            parte = MIMEBase("application", "pdf")
            parte.set_payload(archivo.read())
        
        # Codificar el archivo en base64
        encoders.encode_base64(parte)
        
        # Añadir header al adjunto
        parte.add_header(
            "Content-Disposition",
            f"attachment; filename= {nombre_archivo}",
        )
        
        message.attach(parte)
        
        # Enviar el correo
        print(f"   📧 Conectando al servidor SMTP...")
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, destinatario_email, message.as_string())
        
        print(f"   ✅ Correo enviado exitosamente a {destinatario_email}")
        return True, ""
        
    except FileNotFoundError:
        error = f"Archivo no encontrado: {ruta_pdf}"
        print(f"   ❌ {error}")
        return False, error
        
    except smtplib.SMTPAuthenticationError:
        error = "Error de autenticación. Verifica tu email y contraseña de aplicación"
        print(f"   ❌ {error}")
        return False, error
        
    except smtplib.SMTPException as e:
        error = f"Error SMTP: {str(e)}"
        print(f"   ❌ {error}")
        return False, error
        
    except Exception as e:
        error = f"Error inesperado: {str(e)}"
        print(f"   ❌ {error}")
        return False, error


# === FUNCIÓN PRINCIPAL ===
def main():
    """Función principal que procesa el Excel y envía los correos."""
    
    print("="*60)
    print("   SISTEMA DE ENVÍO DE DESPRENDIBLES DE PAGO POR EMAIL")
    print("="*60)
    
    # Cargar el Excel
    try:
        df = pd.read_excel(ruta_excel)
        print(f"\n✅ Archivo Excel cargado correctamente")
        print(f"   Total de registros: {len(df)}")
        
        # Filtrar solo los pendientes
        pendientes = df[df[columna_estado].str.lower() == "pendiente"]
        print(f"   Empleados pendientes: {len(pendientes)}")
        
        if len(pendientes) == 0:
            print("\n✅ No hay empleados pendientes de envío.")
            return
            
    except FileNotFoundError:
        print(f"❌ No se encontró el archivo Excel: {ruta_excel}")
        return
    except Exception as e:
        print(f"❌ Error al leer el archivo Excel: {e}")
        return
    
    # Verificar que existan las columnas necesarias
    columnas_requeridas = ["empleado", "email", "link", columna_estado]
    columnas_faltantes = [col for col in columnas_requeridas if col not in df.columns]
    
    if columnas_faltantes:
        print(f"❌ Faltan las siguientes columnas en el Excel: {', '.join(columnas_faltantes)}")
        print(f"   Columnas disponibles: {', '.join(df.columns)}")
        return
    
    print("\n" + "="*60)
    print("   INICIANDO ENVÍO DE CORREOS")
    print("="*60 + "\n")
    
    # Procesar cada empleado pendiente
    enviados_exitosos = 0
    errores = 0
    
    for i, row in pendientes.iterrows():
        empleado = str(row["empleado"]).strip().capitalize()
        email = str(row["email"]).strip().lower()
        ruta_pdf = str(row["link"]).strip()
        fecha = str(row['desde']) if 'desde' in row else ''
        fecha_fin = str(row['hasta']) if 'hasta' in row else ''
        if fecha!='' and fecha_fin!='':
            fecha = pd.to_datetime(fecha)
            fecha_fin = pd.to_datetime(fecha_fin)
            fecha = fecha.strftime('%d').capitalize()
            fecha_fin = fecha_fin.strftime('%d de %B de %Y').capitalize()
            fecha = f"{fecha} al {fecha_fin}"
        elif fecha!='':
            fecha = pd.to_datetime(fecha)
            fecha = fecha.strftime('%d de %B de %Y').capitalize()
        else:
            fecha = "Periodo de pago"


        print(f"\n📧 Procesando: {empleado}")
        print(f"   Email: {email}")
        print(f"   PDF: {os.path.basename(ruta_pdf)}")
        
        # Validar email
        if "@" not in email or "." not in email:
            print(f"   ⚠️ Email inválido: {email}")
            df.at[i, columna_estado] = "Error: Email inválido"
            errores += 1
            continue
        
        # Validar que existe el archivo
        if not os.path.exists(ruta_pdf):
            print(f"   ⚠️ Archivo no encontrado: {ruta_pdf}")
            df.at[i, columna_estado] = "Error: Archivo no existe"
            errores += 1
            continue
        
        # Enviar el correo
        exito, mensaje_error = enviar_correo_con_pdf(email, empleado, ruta_pdf, fecha)
        
        if exito:
            df.at[i, columna_estado] = "Enviado"
            enviados_exitosos += 1
        else:
            df.at[i, columna_estado] = f"Error: {mensaje_error[:50]}"
            errores += 1
    
    # Guardar los resultados
    try:
        df.to_excel(ruta_excel, index=False)
        print("\n" + "="*60)
        print("   RESUMEN DEL PROCESO")
        print("="*60)
        print(f"✅ Correos enviados exitosamente: {enviados_exitosos}")
        print(f"❌ Errores: {errores}")
        print(f"✅ Excel actualizado: {ruta_excel}")
        print("="*60 + "\n")
        
    except Exception as e:
        print(f"\n❌ Error al guardar el archivo Excel: {e}")


# === EJECUTAR EL PROGRAMA ===
if __name__ == "__main__":
    main()

   SISTEMA DE ENVÍO DE DESPRENDIBLES DE PAGO POR EMAIL

✅ Archivo Excel cargado correctamente
   Total de registros: 1
   Empleados pendientes: 1

   INICIANDO ENVÍO DE CORREOS


📧 Procesando: William castro
   Email: wcastro@lapocion.com
   PDF: William.pdf
   📧 Conectando al servidor SMTP...
   ✅ Correo enviado exitosamente a wcastro@lapocion.com

   RESUMEN DEL PROCESO
✅ Correos enviados exitosamente: 1
❌ Errores: 0
✅ Excel actualizado: C:\Users\Dataa\Desktop\empleados.xlsx



In [ ]:
import os
import smtplib
import pandas as pd
import locale
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.image import MIMEImage
from email import encoders

# === CONFIGURACIÓN DE IDIOMA ===
try:
    locale.setlocale(locale.LC_TIME, 'es_CO.utf8')  # Español Colombia
except:
    locale.setlocale(locale.LC_TIME, 'es_ES.utf8')  # Español España

# === CONFIGURACIÓN DE CORREO ===
SENDER_EMAIL = "wcastro@lapocion.com"
SENDER_PASSWORD =  # Contraseña de aplicación Gmail
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465

# === ARCHIVOS ===
ruta_excel = r"C:\Users\Dataa\Desktop\empleados.xlsx"
columna_estado = "estado"
ruta_imagen_banner = r"C:\Users\Dataa\Desktop\Final.png"  # ⚠️ Ruta de la imagen a mostrar en el correo


# === FUNCIÓN PARA ENVIAR CORREO CON IMAGEN Y PDF ===
def enviar_correo_con_pdf(destinatario_email, nombre_empleado, ruta_pdf, fecha):
    try:
        # Crear mensaje
        message = MIMEMultipart("related")
        message["From"] = SENDER_EMAIL
        message["To"] = destinatario_email
        message["Subject"] = f"Desprendible de Pago {fecha} - {nombre_empleado}"

        # --- Parte HTML con la imagen inline ---
        cuerpo_html = f"""
        <html>
            <body style="margin:0; padding:0; background-color:#000;">
                <div style="width:100%; height:100%; text-align:center;">
                    <img src="cid:banner_pocion" alt="¡Sííí pagaron!" 
                         style="width:100%; height:auto; display:block; margin:0; padding:0; border:none;"/>
                </div>
            </body>
        </html>
        """
     

        mensaje_html = MIMEText(cuerpo_html, "html", "utf-8")
        message.attach(mensaje_html)

        # --- Adjuntar la imagen en el cuerpo (inline) ---
        with open(ruta_imagen_banner, "rb") as img:
            imagen = MIMEImage(img.read())
            imagen.add_header("Content-ID", "<banner_pocion>")
            imagen.add_header("Content-Disposition", "inline", filename=os.path.basename(ruta_imagen_banner))
            message.attach(imagen)

        # --- Adjuntar el PDF ---
        with open(ruta_pdf, "rb") as archivo:
            adjunto = MIMEBase("application", "pdf")
            adjunto.set_payload(archivo.read())
        encoders.encode_base64(adjunto)
        adjunto.add_header("Content-Disposition", f"attachment; filename={os.path.basename(ruta_pdf)}")
        message.attach(adjunto)

        # --- Enviar correo ---
        print(f"   📧 Enviando correo a {destinatario_email}...")
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, destinatario_email, message.as_string())

        print(f"   ✅ Correo enviado exitosamente a {destinatario_email}")
        return True, ""

    except Exception as e:
        error = f"❌ Error al enviar correo: {str(e)}"
        print(error)
        return False, error


# === FUNCIÓN PRINCIPAL ===
def main():
    print("="*60)
    print("  SISTEMA DE ENVÍO DE DESPRENDIBLES DE PAGO - LA POCIÓN")
    print("="*60)

    try:
        df = pd.read_excel(ruta_excel)
        pendientes = df[df[columna_estado].str.lower() == "pendiente"]
    except Exception as e:
        print(f"❌ Error al leer Excel: {e}")
        return

    if pendientes.empty:
        print("✅ No hay empleados pendientes de envío.")
        return

    enviados_exitosos, errores = 0, 0

    for i, row in pendientes.iterrows():
        empleado = str(row["empleado"]).strip().capitalize()
        email = str(row["email"]).strip().lower()
        ruta_pdf = str(row["link"]).strip()
        fecha = str(row['desde']) if 'desde' in row else ''
        fecha_fin = str(row['hasta']) if 'hasta' in row else ''
        if fecha!='' and fecha_fin!='':
            fecha = pd.to_datetime(fecha)
            fecha_fin = pd.to_datetime(fecha_fin)
            fecha = fecha.strftime('%d').capitalize()
            fecha_fin = fecha_fin.strftime('%d de %B de %Y').capitalize()
            fecha = f"{fecha} al {fecha_fin}"
        elif fecha!='':
            fecha = pd.to_datetime(fecha)
            fecha = fecha.strftime('%d de %B de %Y').capitalize()
        else:
            fecha = "Periodo de pago"

        if fecha:
            fecha = pd.to_datetime(fecha).strftime('%d de %B de %Y').capitalize()
        else:
            fecha = "Periodo de pago"

        print(f"\n📨 Procesando {empleado} ({email})")

        if not os.path.exists(ruta_pdf):
            print(f"   ⚠️ No se encuentra el PDF: {ruta_pdf}")
            df.at[i, columna_estado] = "Error: No se encontró el PDF"
            errores += 1
            continue

        exito, msg = enviar_correo_con_pdf(email, empleado, ruta_pdf, fecha)
        df.at[i, columna_estado] = "Enviado" if exito else f"Error: {msg}"
        enviados_exitosos += exito
        errores += not exito

    df.to_excel(ruta_excel, index=False)
    print(f"\n✅ Correos enviados: {enviados_exitosos} | ❌ Errores: {errores}")
    print(f"📁 Excel actualizado en: {ruta_excel}")


if __name__ == "__main__":
    main()


  SISTEMA DE ENVÍO DE DESPRENDIBLES DE PAGO - LA POCIÓN

📨 Procesando William castro (wcastro@lapocion.com)
   📧 Enviando correo a wcastro@lapocion.com...
   ✅ Correo enviado exitosamente a wcastro@lapocion.com

✅ Correos enviados: 1 | ❌ Errores: 0
📁 Excel actualizado en: C:\Users\Dataa\Desktop\empleados.xlsx
